# Diagonalisation exacte

## I. Fonctions 

### a) Récapitulatif des fonctions utilisées

### b) Le code

In [2]:
# import des modules 

import numpy as np # pour les matrices denses 
import scipy.sparse as sp # pour les matrices sparse (creuses)
import matplotlib.pyplot as plt
from qat.core import Term
from qat.fermion.hamiltonians import FermionHamiltonian
from qat.fermion.hamiltonians import make_anderson_model
from qat.lang import *
from qat.core import Term, Schedule, Variable, Observable
from qat.qpus import get_default_qpu
from qat.plugins import ScipyMinimizePlugin
from qat.lang.AQASM import Program, QRoutine
import numpy.typing as npt
from collections import deque

ModuleNotFoundError: No module named 'qat'

In [ ]:
# définition du Hamiltonien du modèle de Hubbard et des opérateurs associés

def matrice_creation(nqbits: int, i: int, spin: str):
    """ Calcule la matrice de l'opérateur c_{i, spin}.
    """
    if spin=="+":
        M = FermionHamiltonian(nqbits, [Term(1., "C", [i])]).to_spin().get_matrix()
        return M
    else:
        M = FermionHamiltonian(nqbits, [Term(1., "C", [i+1])]).to_spin().get_matrix()
        return M

def matrice_annihilation(nqbits: int, i: int, spin: str):
    """ Calcule la matrice de l'opérateur c^{dagger}_{i, spin}.
    """
    if spin=="+":
        M = FermionHamiltonian(nqbits, [Term(1., "c", [i])]).to_spin().get_matrix()
        return M
    else:
        M = FermionHamiltonian(nqbits, [Term(1., "c", [i+1])]).to_spin().get_matrix()
        return M

def couplingOp(n_qubits: int, energy: float, i: int, j: int, spin_i: str, spin_j: str) : 
    """ Calcule une instance de FermionHamiltonian associée à l'opérateur c^{dagger}_{i, sigma1}c_{j, sigma2}, 
    i et j sont les numéros de site, energy est le préfacteur énergétique apparaissant dans 
    la définition du hamiltonien. 
    """
    if spin_i == "+" :
         if spin_j == "-" :
            return FermionHamiltonian(n_qubits, [Term(energy, "Cc", [i, j+1])])
         else :
            return FermionHamiltonian(n_qubits, [Term(energy, "Cc", [i, j])])
    else :
         if spin_j == "-" :
            return FermionHamiltonian(n_qubits, [Term(energy, "Cc", [i+1, j+1])])
         else :
            return FermionHamiltonian(n_qubits, [Term(energy, "Cc", [i+1, j])]) 
        
def nOp(n_qubits, energy, i, spin) :
    """ Calcule une instance de FermionHamiltonian associée à l'opérateur n_{i, spin} avec le préfacteur energy.
    """
    return couplingOp(n_qubits: int, energy: float, i: int, i: int, spin: str, spin: str)
    
def hamiltonian(n_impurities: int, n_bain: int, U: float, mu: float, t: list, V, epsilon: list) :
    """ Calcule le hamiltonien du modèle d'Anderson et le renvoie sous la forme 
    du couple (SpinHamiltonian, matrice associée).

    U : énergie de répulsion coulombienne (positive)

    t : énergies cinétiques de couplage entre impuretés (négatives)

    mu : potentiel chimique (coût énergétique pour rajouter un électron au système ouvert) (négatif)

    V : énergie cinétique d'hybridation (taille n_impurities * n_bain) (négatives)
    
    epsilon : énergies cinétiques de couplage entre électrons du bain (taille n_bain) (négatives)
    """
    n_qubits = 2 * n_bain + 2 * n_impurities
    H_repulsion = 0
    H_hopping = 0
    H_chem = 0
    for wire in range(0, 2*n_impurities, 2) :
        #Calcul du terme de potentiel chimique
        H_chem += nOp(n_qubits, -mu, wire, "+")
        H_chem += nOp(n_qubits, -mu, wire, "-")

        #Calcul du terme d'effet tunnel entre impuretés, dit "hopping"
        for wire2 in range(0, 2*n_impurities, 2) :
            if wire2 != wire :
                H_hopping += couplingOp(n_qubits, -t, wire, wire2, "+", "-")
                H_hopping += couplingOp(n_qubits, -t, wire, wire2, "+", "+")
                H_hopping += couplingOp(n_qubits, -t, wire, wire2, "-", "-")
                H_hopping += couplingOp(n_qubits, -t, wire, wire2, "-", "+")
            else :
                #Calcul du terme de répulsion coulombienne
                H_repulsion += nOp(n_qubits, U, wire, "+")*nOp(n_qubits, 1, wire,"-")
    
    H_impurity_bath = 0
    H_bath = 0
    for wire3 in range(2*n_impurities, n_qubits, 2) :
        i = (wire3 - 2*n_impurities) // 2
        e = epsilon[i]
        #Calcul du terme lié au bain
        H_bath += nOp(n_qubits, e, wire3, "+")
        H_bath += nOp(n_qubits, e, wire3, "-")
        
        for wire4 in range(0, 2*n_impurities, 2) :
            #Calcul du terme d'interaction impureté-bain
            j = wire4 // 2
            v = V[j][i]
            
            H_impurity_bath += couplingOp(n_qubits, v, wire4, wire3, "+", "+") 
            H_impurity_bath += couplingOp(n_qubits, v, wire3, wire4, "+", "+")
            
            v = np.conj(v)
            
            H_impurity_bath += couplingOp(n_qubits, v, wire4, wire3, "-", "-") 
            H_impurity_bath += couplingOp(n_qubits, v, wire3, wire4, "-", "-")
        

    H_anderson = H_repulsion + H_chem + H_impurity_bath + H_bath + H_hopping
    H_spin = H_anderson.to_spin(method = "jordan-wigner")#Fermion to Qubit
    return H_spin, H_spin.get_matrix()

In [3]:
# algorithme de Lanczos

def lanczos(init: np.ndarray[float], H, m: int, eps: float):
    """ Renvoie la liste des coefficients diagonaux et celle des 
    coefficients sous-diagonaux et surdiagonaux de la matrice de 
    Lanczos 
    
    init : vecteur initial
    
    H : matrice à diagonaliser (préférablement au format csc_array)
    
    m : nombre d'itérations de l'algorithme
    
    eps : seuil d'acceptation
    """
    a = []
    b = []
    v = init
    V = [init]
    b0 = np.linalg.norm(v)
    b.append(b0)
    v = v/b0  ## Renormalisation, éventuellement dispensable
    w = np.zeros_like(v)
    w += H@v
    a.append((v.T)@w)
    w = w - a[0]*v
    b.append(np.linalg.norm(w))
    for i in range(1,m):
        if b[-1]<eps :
            print("b_i = 0, abort")
            break
        v_new = w/b[-1]  #v_j = w_{j-1} / b_j
        V.append(v_new)
        w = H@v_new
        a.append(v_new.T@w)
        w = w - a[-1]*v_new - b[-1] * v
        v = v_new
        b.append(np.linalg.norm(w))
    V = np.array(V)
    
    return a, b[:-1], V.T

TypeError: Type subscription requires python >= 3.9

In [ ]:
# créer le hamiltonien tridiagonal T

def tridiagonale(diag: np.ndarray[float],sousdiag: np.ndarray[float],surdiag: np.ndarray[float]):
    """ Crée une matrice tridiagionale avec les coefficients correspondants
    """
    return np.diag(diag) + np.diag(sousdiag, k =-1) + np.diag(surdiag, k =1)

In [4]:
# fonction de Green selon la formule de Lehmann

def Green_Lehmann(valeurs_prop: np.ndarray[float], vecteurs_prop: np.ndarray[float], n_site_bain, Omega, eta):
    n_site = n_site_bain + 1
    n_qbits = 2 * n_site
    dim, nb_vp = np.shape(vecteurs_prop)
    
    mat_crea= matrice_creation(n_qbits, 0, "+")
    mat_nihil = matrice_annihilation(n_qbits, 0, "+")
    
    N = len(Omega)
    valeurs_prop = valeurs_prop - valeurs_prop[0]
    S = []
    for i in range(len(Omega)):
        S_i = 0
        for n in range(nb_vp):
            prod_scal1 = vecteurs_prop[0].T @ mat_crea @ vecteurs_prop[n]
            prod_scal1 = np.real(prod_scal1)**2 + np.imag(prod_scal1)** 2
            S_i += ( prod_scal1 ) / (
                       Omega[i] + valeurs_prop[n] - valeurs_prop[0]  - 1j * eta)
            
            prod_scal2 = vecteurs_prop[0].T @ mat_nihil @ vecteurs_prop[n]
            prod_scal2 = np.real(prod_scal2)**2 + np.imag(prod_scal2)** 2
            S_i += ( prod_scal2 ) / (
                         Omega[i] + valeurs_prop[0] - valeurs_prop[n] + 1j * eta )
        S.append(S_i)
    return S

In [ ]:
# fonction spectrale selon la formule de Lehmann
def A_Lehman(valeurs_prop: np.ndarray[float], vecteurs_prop: np.ndarray[float], n_site_bain, Omega, eta):
    return - 1 / np.pi * np.imag(Green_Lehmann_zero_abs(valeurs_prop, vecteurs_prop, n_site_bain, Omega, eta)) ## PARTIE IMAGINAIRE
    

In [ ]:
# fonction de Green approchée avec l'algorithme de Lanczos

def param_Green(init,H, m, eps):
    """ Calcule l'énergie initiale E_0 et les états initiaux psi_c = cdagger_0 psi_0 et psi_d = c_0 psi_0 pour le calcul de la fonction de Green 
    avec excitation particulaire et excitation lacunaire respectivement"""
    a,b,V = lanczos_v2(init,H,m,eps)
    T = tridiagonale(a,b[1:],b[1:]) # b_0 n'apparaît pas dans la matrice tridiagonale
    
    cdagger_0 = matrice_creation(nqbits, 0, "+")
    c_0 = matrice_annihilation(nqbits, 0, "+")
    
    E_0, F = sp.linalg.eigs(T, which = 'SR', k = 1)
    F = V@F
    return (E_0, cdagger_0@F, c_0@F)
    
def greenLanczos(Omega, Eta, psi_c, psi_d, E_0, H, m, eps):
    """Calcule la fonction de Green approchée par 
    l'algortihme de Lanczos"""
    a_c, b_c, V_c = lanczos_v2(psi_c, H, m, eps)
    a_d, b_d, V_d = lanczos_v2(psi_d, H, m, eps)
    return ( greenLanczosAdapt(Omega  + 1j * Eta + E_0 * np.ones_like(Omega), a_c, b_c) - 
    greenLanczosAdapt(- Omega - 1j * Eta + E_0 * np.ones_like(Omega), a_d, b_d) )

def greenLanczosFrac(z: np.ndarray[float], a: np.ndarray[float], b: np.ndarray[float]):
    """Calcule l'image de Z par la transformée de Fourier approchée 
    de la fonction de Green
    """
    frac = np.zeros_like(z)
    n = len(b)
    for i in range(n-1, 0, -1):
        frac = b[i]**2/(z-a[i]-frac)
    return 1/(z-a[0]-frac)

In [ ]:
#Fonction spectrale par Lanczsos
def spectraleLanczos(Omega, Eta, psi_c, psi_d, E_0, H, m, eps):
    """ Calcule la fonction spectrale approchée par l'algorithme de Lanczos"""
    return - 1/np.pi * np.imag(greenLanczos(Omega, Eta, psi_c, psi_d, E_0, H, m, eps)) # PARTIE IMAGINAIRE
    

## II. Tests

### a) Convergence de l'algorithme de Lanczos

In [1]:
n_impur = 1 
n_bain = 1
nqbits = 2 * (n_impur + n_bain)
N_sys = 4**(n_impur + n_bain)

low_U = 1#0
high_U = 1#10
U = np.random.uniform(low_U,high_U) # Coulombian repulsion

low_mu = 1#0
high_mu = 1#10
mu = np.random.uniform(low_mu,high_mu) # Chemical potential

low_T = -1#-10
high_T = -1#0
T = np.random.uniform(low_T,high_T) # Hopping between impurities


low_V = -1#-10
high_V = -1#0
V_alea = np.random.uniform(low_V, high_V, (n_impur,n_bain)) # Hybridization energy

low_E = -1#-10
high_E = -1#0
Eps_alea = np.random.uniform(low_E,high_E, n_bain) # Hopping of conduction electrons


print(U,mu,T,V_alea,Eps_alea)


AndersonHamiltonian, H = hamiltonian(n_impur, n_bain, U, mu, T, V_alea, Eps_alea)

NameError: name 'np' is not defined

Nous allons contrôler dans un premier temps la convergence de la valeur propre la plus faible.

In [ ]:
def ecartVPLower(H, m: int, init: np.ndarray[float], eps: float):
    """ Affiche l'écart absolu entre les valeurs propres théoriques et pratiques la plus faible.
    """
    ecart = []
    iteration = []
    eigvallow = np.real(sp.linalg.eigs(H,  which = 'SR', k = 1)[0][0])
    for i in range(1,m):
        iteration.append(i)
        a, b, V = lanczos(init, H, m, eps)
        T = tridiagonale(a, b[1:], b[1:])
        ecart.append(abs(np.linalg.eigvalsh(T)[0]-eigvallow))
    plt.figure()
    plt.plot(iteration, ecart, label = "écart absolu")
    plt.title("Convergence de la valeur propre la plus faible via l'algorithme de Lanczos")
    plt.xlabel("Itération")
    plt.ylabel("Écart absolu entre la valeur propre théorique la plus faible et celle obtenue par l'algorithme")
    plt.show()
    return

Nous comparons à chaque itération la nouvelle valeur de b calculée à epsilon, mais est-ce que b converge ?

In [ ]:
def affichage(H, m: int, init: np.ndarray[float], eps: float):
    a, b, V = lanczos(init, H, m, eps)
    plt.figure()
    plt.plot(b, label = "Évolution du paramètre b")
    plt.xlabel("Itération")
    plt.ylabel("Grandeur b")
    plt.show()
    return

### b) Comparaison des fonctions de Green de Lehmann et celle approchée par Lanczos

## III. Théorie